In [ ]:

import os
import pandas as pd 
import os

from plotting import trace_plots as p_trace
from conversions import trace_conversions as c_trace

from analysis_util import *

from common_constants import *
import filters.trace_filters as f_trace
import plotly.io as pio

pio.templates.default = "seaborn"

update_figures =  False
images_folder = "../images"

dataset_path = f"../dataset/fragverif_parsed/conversion/run_0"                          # Dataset path for the benchmark_parsed dataset. This is where the parquet files are located.
query_path = os.path.join(dataset_path, f"dataset_query_search.parquet")                # Path to the parquet file containing the query search data.
trace_path = os.path.join(dataset_path, f"dataset_trace_actions.parquet")               # Path to the parquet file containing the trace actions data.
conv_trace_path = os.path.join(dataset_path, f"dataset_conv_trace_actions.parquet")     # Path to the parquet file containing the converted trace actions data.
hashes_path = os.path.join(dataset_path, 'hashed_traces')                               # Path to the directory containing the hashed traces. 

df_query_search = pd.read_parquet(query_path)
df_query_search_copy = df_query_search.copy()
df_actions = pd.read_parquet(trace_path)
df_conv_actions = pd.read_parquet(conv_trace_path).merge(df_query_search[['trace_group_identifier', 'trace_identifier', 'hash_trace']], on='hash_trace', how='left')


In [3]:
common_limits = [
    0,
    10,
    25,
    50,
    75,
]

limit_point_thresholds = [
    c_trace.PointThreshold(0, 1),
    c_trace.PointThreshold(10, 0.75),
    c_trace.PointThreshold(25, 0.50),
    c_trace.PointThreshold(50, 0.25),
    c_trace.PointThreshold(75, 0.0)
]

      

In [ ]:
from math import ceil, floor, log
from common_constants import TraceConstants as tc
import filters.trace_filters as t_f
import filters.model_filters as m_f
import common_constants as cc
from conversions.trace_conversions import LIMIT_SEPARATOR

# Compare each step within the limits, and then compare the overall points.
def compare_steps(df_frag: pd.DataFrame, df_points: pd.DataFrame, frag_s: str, exp_values: list[int], limits: list[int], thresholds: list[c_trace.PointThreshold]):
    total_points = 0
    valuations = 0
    for (i, l) in enumerate(limits):
        lim_s = frag_s + LIMIT_SEPARATOR + str(l)
        val_frag = df_frag[lim_s].loc[0]
        assert val_frag == exp_values[i]

        total_points += thresholds[i].valuation * exp_values[i]
        valuations += exp_values[i]

    # +/- 0.002 point deviation taking into account floats.
    val_points = round(df_points[frag_s].loc[0] * 100, 1)
    exp_points = round(total_points / valuations * 100, 1)
    assert val_points == exp_points


def compare_frag(tg: TraceGroupData, base: int, multiple: int, page_size: int, limits: list[int], thresholds: list[c_trace.PointThreshold], num_allocs_peak:int = 200):
    peaks_fact = [1, 2, 4, 2, 4, 2, 1]
    max_peak_mult = max(peaks_fact)

    # Get fragverif traces with the given base and multiple, and apply the model filter for FFAO.
    filter = t_f.get_tg(['fragverif_trace']) \
    .merge([t_f.get_trace_num([str(base)])]) \
    .merge([m_f.MODEL_FILTER_POLICY_FFAO])\
    .merge([m_f.get_size_multiple(multiple)])

    df_frag = filter.apply(tg.df_model_frag)
    df_points = filter.apply(tg.df_model_points)

    # We use power of 2's so 200->256, etc.
    peak_rounded_base = multiple * ceil(max_peak_mult * base / multiple)
    exp_max_peak_page = ceil((peak_rounded_base * num_allocs_peak) / page_size) * page_size
    exp_max_peak_alloc = peak_rounded_base * num_allocs_peak
    exp_max_peak_req = max_peak_mult * base * num_allocs_peak
    
    # Multiple by two because balanced trace, so #allocs=#frees
    num_total_actions = len(peaks_fact) * 2 * num_allocs_peak

    # Create lists to hold the total number of actions that exceed each limit for each fragmentation metric.
    total_frag_lfb = [0 for _ in limits]
    total_frag_rel_ext_PV = [0 for _ in limits]
    total_frag_rel_int = [0 for _ in limits]
    reverse_lims = limits[::-1]

    # Compare LFB_PV
    for action in range(0, num_total_actions):
        peak_num = floor(action / (num_allocs_peak*2))
        peak_mult = peaks_fact[peak_num]
        action_peak = action - (peak_num*num_allocs_peak*2) + 1
        rounded_base = multiple * ceil(peak_mult * base / multiple)

        # Rising edge
        if action_peak <= num_allocs_peak:
            exp_peak_page = ceil((rounded_base * action_peak) / page_size) * page_size
            exp_peak_alloc = rounded_base * action_peak
            exp_peak_req = peak_mult * base * action_peak

            exp_peak_padded = exp_peak_alloc - exp_peak_req
            exp_free_mem = exp_peak_page - exp_peak_alloc
            ext_peak_lfb = exp_peak_page - exp_peak_alloc
        # Falling edge
        else:
            exp_local_peak_page = ceil((rounded_base * num_allocs_peak) / page_size) * page_size
            exp_local_peak_alloc = rounded_base * num_allocs_peak
            exp_local_peak_req = peak_mult * base * num_allocs_peak
            num_free = action_peak - num_allocs_peak
            # Expected peaks for page, allocated memory, and requested memory.
            exp_peak_page = exp_local_peak_page if num_free % 200 != 0 else 0
            exp_peak_alloc = exp_local_peak_alloc - (rounded_base * num_free)
            exp_peak_req = exp_local_peak_req - (peak_mult * base * num_free)

            exp_peak_padded = exp_peak_alloc - exp_peak_req
            exp_free_mem = exp_peak_page - exp_peak_alloc
            ext_peak_lfb = max(rounded_base * num_free, exp_local_peak_page - exp_local_peak_alloc) # Either block between last alloc and end of last page, or the block starting from 0x0 up to next alloc to be freed (FIFO)
        # Calculate the expected fragmentation metrics for this action.
        exp_peak_frag_lfb = (100 - ext_peak_lfb * 100 // exp_free_mem) if exp_free_mem != 0 else 0
        exp_peak_frag_rel_ext_PV = (100 - exp_peak_alloc * 100 // exp_peak_page) if exp_peak_page != 0 else 0
        exp_peak_frag_rel_int = (exp_peak_padded * 100 // exp_peak_alloc) if exp_peak_alloc != 0 else 0

        found_lfb = found_ext = found_int = False
        for (i, frag) in enumerate(reverse_lims):
            lim_index = len(limits) - 1 - i
            if not found_lfb and exp_peak_frag_lfb >= frag:
                total_frag_lfb[lim_index] += 1
                found_lfb = True

            if not found_ext and exp_peak_frag_rel_ext_PV >= frag:
                total_frag_rel_ext_PV[lim_index] += 1
                found_ext = True

            if not found_int and exp_peak_frag_rel_int >= frag:
                total_frag_rel_int[lim_index] += 1
                found_int = True

            if found_lfb and found_ext and found_int:
                break

    compare_steps(df_frag, df_points, cc.ExtFragConstants.LFB_PV.value, total_frag_lfb, limits, thresholds)
    compare_steps(df_frag, df_points, cc.ExtFragConstants.Relative_PV.value, total_frag_rel_ext_PV, limits, thresholds)
    compare_steps(df_frag, df_points, cc.IntFragConstants.relative.value, total_frag_rel_int, limits, thresholds)

    # ------- FRAG LINEAR-------
    # Compare Int WJ max 
    val_frag_int_wj_max = df_frag[cc.IntFragConstants.WJ_max.value].loc[0]
    exp_frag_int_wj_max = floor(exp_max_peak_alloc/exp_max_peak_req * 100)
    assert val_frag_int_wj_max == exp_frag_int_wj_max

    # Compare Int WJ reserved upper 
    val_frag_int_wj_reserved_upper = df_frag[cc.IntFragConstants.WJ_max.value].loc[0]
    exp_frag_int_wj_reserved_upper = floor(exp_max_peak_alloc/exp_max_peak_req * 100)
    assert val_frag_int_wj_reserved_upper == exp_frag_int_wj_reserved_upper

    # Compare WJ alloc reserved upper
    val_frag_wj_alloc_reserved_upper = df_frag[cc.ExtFragConstants.WJ_alloc_reserved_upper.value].loc[0]
    exp_frag_wj_alloc_reserved_upper = floor(exp_max_peak_page/exp_max_peak_alloc * 100)
    assert val_frag_wj_alloc_reserved_upper == exp_frag_wj_alloc_reserved_upper

    # Compare WJ req reserved upper
    val_frag_wj_req_reserved_upper = df_frag[cc.ExtFragConstants.WJ_req_reserved_upper.value].loc[0]
    exp_frag_wj_req_reserved_upper = floor(exp_max_peak_page/exp_max_peak_req * 100)
    assert val_frag_wj_req_reserved_upper == exp_frag_wj_req_reserved_upper

    # Compare WJ alloc max
    val_frag_wj_alloc_max = df_frag[cc.ExtFragConstants.WJ_alloc_max.value].loc[0]
    exp_frag_wj_alloc_max = floor(exp_max_peak_page/exp_max_peak_alloc * 100)
    assert val_frag_wj_alloc_max == exp_frag_wj_alloc_max

    # Compare WJ req max
    val_frag_wj_req_max = df_frag[cc.ExtFragConstants.WJ_req_max.value].loc[0]
    exp_frag_wj_req_max = floor(exp_max_peak_page / exp_max_peak_req * 100)
    assert val_frag_wj_req_max == exp_frag_wj_req_max

    # ------- POINTS LINEAR -------
    # Compare Int WJ reserved upper 
    val_points_int_wj_reserved_upper = df_points[cc.IntFragConstants.WJ_max.value].loc[0]
    exp_points_int_wj_reserved_upper = 1 - (exp_frag_int_wj_max - 100) / 200
    assert val_points_int_wj_reserved_upper == exp_points_int_wj_reserved_upper

    # Compare WJ alloc reserved upper
    val_points_wj_alloc_reserved_upper = df_points[cc.ExtFragConstants.WJ_alloc_reserved_upper.value].loc[0]
    exp_points_wj_alloc_reserved_upper = 1 - (exp_frag_wj_alloc_reserved_upper - 100) / 200
    assert val_points_wj_alloc_reserved_upper == exp_points_wj_alloc_reserved_upper

    # Compare WJ req reserved upper
    val_points_wj_req_reserved_upper = df_points[cc.ExtFragConstants.WJ_req_reserved_upper.value].loc[0]
    exp_points_wj_req_reserved_upper = 1 - (exp_frag_wj_req_reserved_upper - 100) / 200
    assert val_points_wj_req_reserved_upper == exp_points_wj_req_reserved_upper

    # Compare WJ alloc max
    val_points_wj_alloc_max = df_points[cc.ExtFragConstants.WJ_alloc_max.value].loc[0]
    exp_points_wj_alloc_max = 1 - (exp_frag_wj_alloc_max - 100) / 200
    assert val_points_wj_alloc_max == exp_points_wj_alloc_max

    # Compare WJ req max
    val_points_wj_req_max = df_points[cc.ExtFragConstants.WJ_req_max.value].loc[0]
    exp_points_wj_req_max = 1 - (exp_frag_wj_req_max - 100) / 200
    assert val_points_wj_req_max == exp_points_wj_req_max


tg = process_tg(df_query_search, tc.tg_s_fragverif.value, f_trace.TRACE_FILTER_FRAGVERIF, common_limits, limit_point_thresholds, hashes_path)

# Model expected values for each size multiple and base allocation size.
page_size = 4096
num_allocations_per_peak = 200
for sm in [64, 16, 1]:
    for base in [100, 200]:
        compare_frag(tg, base, sm, page_size, common_limits,  limit_point_thresholds, num_allocations_per_peak)